# Adaptive-RAG routed baseline: classifier training + evaluation (RQ1)

Paper: Jeong et al., NAACL 2024 (arXiv:2403.14403).
Code: https://github.com/starsuzi/Adaptive-RAG

Trains the t5-large query-complexity classifier and scores the routed
Adaptive-RAG system on the 6 QA test sets. These are the numbers my zero-shot
LLM router is compared against (RQ1).

Runs on a Colab GPU runtime attached from VS Code. The training cell needs
an A100 namely the authors train t5-large at batch size 32 and sequence
length 384, which OOMs on a T4's 15 GB (tried 17 Aug, died 2 steps in).
Keeping their batch size on a bigger card beats shrinking it, so the
reproduction stays faithful to the paper. Everything is re-created on a
fresh runtime: run the cells top to bottom.

In [1]:
![ -d Adaptive-RAG ] || git clone -q https://github.com/starsuzi/Adaptive-RAG.git
%cd Adaptive-RAG
# record the commit I'm working from, this goes in the thesis write-up
COMMIT = !git rev-parse HEAD
print("using commit:", COMMIT[0])

using commit: 0c88670af8707667eb5c1163151bb5ce61b14acb


In [2]:
%%bash
set -e
# colab exports a PYTHONPATH for its own python 3.12, keep it away from the 3.8 env
unset PYTHONPATH
# upstream needs python 3.8 (torch<2 and an old transformers commit, see requirements.txt).
# colab's default python is too new for those pins, so everything runs through a
# small conda env instead. -u lets the installer rerun over an existing install.
wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
bash /tmp/miniconda.sh -b -u -p /opt/miniconda
# python 3.8 from conda-forge (anaconda's default channels now need a ToS acceptance)
[ -d /opt/miniconda/envs/arag ] || \
  /opt/miniconda/bin/conda create -y -q -n arag -c conda-forge --override-channels python=3.8
/opt/miniconda/envs/arag/bin/pip install -q -r requirements.txt
/opt/miniconda/envs/arag/bin/python -c "import _jsonnet; print('env ok')"

PREFIX=/opt/miniconda
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
env ok


In [3]:
%%bash
# shipped tarballs: per-strategy predictions (the classifier routes between them),
# test subsamples (ground truths), preprocessed classifier training data
tar -xzf predictions.tar.gz
tar -xzf processed_data.tar.gz
tar -xzf data.tar.gz
ls classifier/data | head -3

musique_hotpot_wiki2_nq_tqa_sqd


In [4]:
%%bash
unset PYTHONPATH
# cuda build of the pinned torch
/opt/miniconda/envs/arag/bin/pip install -q torch==1.13.1+cu117 --extra-index-url https://download.pytorch.org/whl/cu117
/opt/miniconda/envs/arag/bin/python -c "import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0))"

True Tesla T4


In [5]:
%%bash
export PATH=/opt/miniconda/envs/arag/bin:$PATH
# the env ships its own libstdc++ and sqlite; without this the loader picks
# colab's older system libstdc++ and nltk's sqlite import dies on CXXABI
export LD_LIBRARY_PATH=/opt/miniconda/envs/arag/lib:${LD_LIBRARY_PATH:-}
unset PYTHONPATH
cd classifier
# two edits before training: the script pins GPU 7 (authors' server, colab has one gpu),
# and it sweeps epochs 15-35, training five times. the paper settled on epoch 25,
# so one training run is enough here.
sed -i 's/^GPU=7/GPU=0/' run/run_large_train_xl.sh
sed -i 's/^for EPOCH in 15 20 25 30 35/for EPOCH in 25/' run/run_large_train_xl.sh
bash run/run_large_train_xl.sh

You're running a t5 model but didn't provide a source prefix, which is the expected, e.g. with `--source_prefix 'summarize: ' `
Generating train split: 3692 examples [00:00, 89468.92 examples/s]
/opt/miniconda/envs/arag/lib/python3.8/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
loading configuration file config.json from cache at /content/Adaptive-RAG/cache/models--t5-large/snapshots/150ebc2c4b72291e770f58e6057481c8d2ed331a/config.json
Model config T5Config {
  "_name_or_path": "t5-large",
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "d_ff": 4096,
  "d_kv": 64,
  "d_model": 1024,
  "decoder_start_token_id": 0,
  "dense_act_fn": "relu",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "relu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": 

CalledProcessError: Command 'b"export PATH=/opt/miniconda/envs/arag/bin:$PATH\n# the env ships its own libstdc++ and sqlite; without this the loader picks\n# colab's older system libstdc++ and nltk's sqlite import dies on CXXABI\nexport LD_LIBRARY_PATH=/opt/miniconda/envs/arag/lib:${LD_LIBRARY_PATH:-}\nunset PYTHONPATH\ncd classifier\n# two edits before training: the script pins GPU 7 (authors' server, colab has one gpu),\n# and it sweeps epochs 15-35, training five times. the paper settled on epoch 25,\n# so one training run is enough here.\nsed -i 's/^GPU=7/GPU=0/' run/run_large_train_xl.sh\nsed -i 's/^for EPOCH in 15 20 25 30 35/for EPOCH in 25/' run/run_large_train_xl.sh\nbash run/run_large_train_xl.sh\n"' returned non-zero exit status 1.

In [ ]:
%%bash
export PATH=/opt/miniconda/envs/arag/bin:$PATH
# the env ships its own libstdc++ and sqlite; without this the loader picks
# colab's older system libstdc++ and nltk's sqlite import dies on CXXABI
export LD_LIBRARY_PATH=/opt/miniconda/envs/arag/lib:${LD_LIBRARY_PATH:-}
unset PYTHONPATH
# both scripts hardcode the authors' timestamped run directory. find the run I just
# trained and point them at it instead.
RESULT=$(ls -t classifier/outputs/*/model/t5-large/flan_t5_xl/epoch/*/*/*/predict/dict_id_pred_results.json | head -1)
echo "using $RESULT"
sed -i "s|^classification_result_file = .*|classification_result_file = './$RESULT'|" classifier/postprocess/predict_complexity_on_classification_results.py
python classifier/postprocess/predict_complexity_on_classification_results.py flan_t5_xl
BASE="predictions/classifier/$(echo "$RESULT" | sed 's|.*/model/||; s|/predict/.*||')/"
sed -i "s|^base_pred_path = .*|base_pred_path = './$BASE'|" evaluate_final_acc.py
python evaluate_final_acc.py

Routed EM/F1 per dataset from the cell above go into the RQ1 comparison table
(zero-shot LLM router vs trained t5-large classifier), alongside routing
accuracy once my router runs on the same test subsamples.